In [1]:
libraries<-c("DESeq2","ggplot2","viridis","RColorBrewer","pheatmap","edgeR","ggfortify","factoextra")
suppressPackageStartupMessages(lapply(libraries, require, character.only = TRUE)) 

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

In [2]:
#Archivo de cuentas
countData <- read.csv("your_path/transcript_count_matrix_agrupada.csv",
                      sep = ',',row.names=1)

In [3]:
# aqui se elimina la columna 1
countData[,1] <- NULL

In [4]:
#Metadata(?)
samples <- read.delim("/your_path/sampledatainfo_hypocotyl2.txt",
                      header=T)

In [5]:
# crear un grupo basado en la metadata, del objeto samples toma la columna treatment
groups<-samples$treatment

In [6]:
# correr DGEList con la data y los grupos, 
#Crea un objeto de clase DGEList, que es la estructura central que usa edgeR para manejar datos de conteo de RNA-seq.
#counts = countData debe ser siempre con filas de genes transcritos y columnas de muestras
#group = factor(groups) le indica a R que los datos de groups son categóricos y no números
d <- DGEList(counts=countData,group=factor(groups))

In [7]:
# Filter lowly expressed genes
keep <- filterByExpr(d)
d <- d[keep, , keep.lib.sizes=FALSE]

In [8]:
#número total de genes y número de muestras
dim(d)

[1] 33255    32

In [9]:
#Esta línea aplica la normalización TMM (Trimmed Mean of M-values) a tu objeto DGEList y genera un nuevo objeto llamado d_TMM.
#Ahora, el objeto d_TMM contiene los factores de normalización ajustados por TMM, 
#que puedes usar para recalcular los CPM de una forma más adecuada para comparaciones entre muestras con composiciones muy diferentes.
#Los factores TMM (no son cuentas o expresión) es una normalización que se basa en el tamaño de la biblioteca y la distribución relativa de los genes entre muestras. 
#Así se corrige por diferencias en la composición del transcriptoma entre las muestras.
d_TMM <- calcNormFactors(d, method="TMM")

In [10]:
#Calculate the cpm with the TMM normalized library
#Ahora el objeto TMM tiene las cuentas en CPM, pero normalizadas con el método TMM
#normalized.lib.sizes = TRUE: edgeR usa lib.sizexnorm.factors como denominador para el calculo de cpms
TMM <- cpm(d_TMM, log = FALSE, normalized.lib.sizes=TRUE)
tmm_log <- log(1+TMM)

In [21]:
# guardar cuentas
#cpms son cuentas sin normalizar
#TMM_son cuentas cpms normalizadas con TMM, sin filtrar genes por expresión todavía
write.table(cpms,file = "/your_path/normCounts_33255_cpms_Lluteus.tsv",sep = '\t',quote = F)
write.table(cmpl,file = "/your_path/normCounts_33255_cpms_log_Lluteus.tsv",sep = '\t',quote = F)
write.table(TMM,file = "/your_path/normCounts_33255_tmm_Lluteus.tsv",sep = '\t',quote = F)
write.table(tmm_log,file = "/your_path/normCounts_33255_tmm_log_Lluteus.tsv",sep = '\t',quote = F)

In [11]:
# Normalize the data, calcNormFactors usa TMM por defecto
d <- calcNormFactors(d)

In [12]:
#Create a design matrix to specify the model.
#Resultado: cada columna representa un grupo experimental distinto, con 1 si la muestra pertenece a ese grupo, y 0 si no.
#con ~groups R incluye un intercepto (grupo de referencia), y las demás columnas representan diferencias respecto a ese grupo.
#~ 0 + groups: cada grupo tiene su propia columna independiente, y vos podés definir contrastes explícitos entre los grupos.
design <- model.matrix(~ 0 + groups)  

In [13]:
#las columnas de design se llamarán como groups
colnames(design) <- levels(as.factor(groups))

In [14]:
# Estimate dispertion de la data para cada gen basado en el diseño
y <- estimateDisp(d, design)

In [15]:
# Fit the model
#Esta función ajusta un modelo lineal generalizado (GLM) para cada gen, usando Quasi-Likelihood Test, más preciso y mayor controld de falsos +
fit <- glmQLFit(y, design)

In [18]:
#fit #contiene Los coeficientes del modelo para cada gen (cómo se expresa en cada grupo)

In [19]:
colnames(design)

[1] "C195_000HPI" "C195_024HPI" "C195_060HPI" "C195_084HPI" "C98_000HPI" 
[6] "C98_024HPI"  "C98_060HPI"  "C98_084HPI"

In [20]:
#makeContrast combinación lineal de coeficientes del modelo, el nombre del contraste = el contraste, Usando las columnas del design como base para los nombres de los grupos
#qlf2: Realiza el test quasi-likelihood F-test, que es más robusto que el likelihood ratio test. Contiene log2, foldchange,p-value,F,etc
#top2:extrae y ordena los resultados por significancia FDR

#EL OOOORDEENNN Es C98_000HPI_vs_C195_000HPI; tratamiento vs control

#makeContrasts(C98_000HPI_vs_C195_000HPI = C98_000HPI - C195_000HPI, levels = design
contrast1 <- makeContrasts(C98_000HPI_vs_C195_000HPI = C98_000HPI - C195_000HPI, levels = design)
qlf1 <- glmQLFTest(fit, contrast = contrast1)
top1 <- topTags(qlf1, n = Inf)


In [21]:
contrast2 <- makeContrasts(C98_024HPI_vs_C195_024HPI = C98_024HPI - C195_024HPI, levels = design)
qlf2 <- glmQLFTest(fit, contrast = contrast2)
top2 <- topTags(qlf2, n = Inf)


In [22]:
contrast3 <- makeContrasts(C98_060HPI_vs_C195_060HPI = C98_060HPI - C195_060HPI, levels = design)
qlf3 <- glmQLFTest(fit, contrast = contrast3)
top3 <- topTags(qlf3, n = Inf)


In [23]:
contrast4 <- makeContrasts(C98_084HPI_vs_C195_084HPI = C98_084HPI - C195_084HPI, levels = design)
qlf4 <- glmQLFTest(fit, contrast = contrast4)
top4 <- topTags(qlf4, n = Inf)


In [28]:
# Save the results to a TSV file
write.table(as.data.frame(top1), file = "/your_path/EDGER/EdgeR_C98_000HPI_vs_C195_000HPI_all.tsv", sep = "\t", quote = FALSE, row.names = TRUE)
write.table(as.data.frame(top2), file = "/your_path/EDGER/EdgeR_C98_024HPI_vs_C195_024HPI_all.tsv", sep = "\t", quote = FALSE, row.names = TRUE)
write.table(as.data.frame(top3), file = "/your_path/EDGER/EdgeR_C98_060HPI_vs_C195_060HPI_all.tsv", sep = "\t", quote = FALSE, row.names = TRUE)
write.table(as.data.frame(top4), file = "/your_path/EDGER/EdgeR_C98_084HPI_vs_C195_084HPI_all.tsv", sep = "\t", quote = FALSE, row.names = TRUE)

In [29]:
#filtro para expresión diferencial
DE_list_C98_000HPI_vs_C195_000HPI <- subset(top1$table, FDR < 0.05 & logFC>= 2 | FDR < 0.05 & logFC <= -2)
DE_list_C98_024HPI_vs_C195_024HPI <- subset(top2$table, FDR < 0.05 & logFC>= 2 | FDR < 0.05 & logFC <= -2)
DE_list_C98_060HPI_vs_C195_060HPI <- subset(top3$table, FDR < 0.05 & logFC>= 2 | FDR < 0.05 & logFC <= -2)
DE_list_C98_084HPI_vs_C195_084HPI <- subset(top4$table, FDR < 0.05 & logFC>= 2 | FDR < 0.05 & logFC <= -2)

In [30]:
# Save the results to a TSV file
write.table(as.data.frame(DE_list_C98_000HPI_vs_C195_000HPI), file = "/your_path/EDGER/EdgeR_C98_000HPI_vs_C195_000HPI_DE.tsv", sep = "\t", quote = FALSE, row.names = TRUE)
write.table(as.data.frame(DE_list_C98_024HPI_vs_C195_024HPI), file = "/your_path/EDGER/EdgeR_C98_024HPI_vs_C195_024HPI_DE.tsv", sep = "\t", quote = FALSE, row.names = TRUE)
write.table(as.data.frame(DE_list_C98_060HPI_vs_C195_060HPI), file = "/your_path/EDGER/EdgeR_C98_060HPI_vs_C195_060HPI_DE.tsv", sep = "\t", quote = FALSE, row.names = TRUE)
write.table(as.data.frame(DE_list_C98_084HPI_vs_C195_084HPI), file = "/your_path/EDGER/EdgeR_C98_084HPI_vs_C195_084HPI_DE.tsv", sep = "\t", quote = FALSE, row.names = TRUE)

In [31]:
#Antes de correr DESeq2, verificar que las columnas de countData_final estén en el mismo orden que las filas de samples
d$counts

,C195_60H_1,C195_60H_2,C195_60H_3,C195_60H_4,C195_84H_1,C195_84H_2,C195_84H_3,C195_84H_4,C195_24H_1,C195_24H_2,⋯,C98_84H_3,C98_84H_4,C98_24H_1,C98_24H_2,C98_24H_3,C98_24H_4,C98_C1,C98_C2,C98_C3,C98_C4
Llu00001,2,5,12,15,24,7,15,5,39,19,⋯,7,14,4,10,10,21,31,20,7,5
Llu00002,1209,1518,1608,1224,1168,1418,1062,1311,1148,799,⋯,880,1252,878,615,855,869,1120,704,1051,1009
Llu00003,568,504,563,341,393,513,511,562,556,331,⋯,428,680,607,341,202,785,797,443,751,820
Llu00004,34,56,28,32,9,21,25,28,39,20,⋯,24,31,14,2,3,19,6,5,14,15
Llu00005,261,248,211,171,171,154,107,129,414,492,⋯,166,295,280,324,258,556,845,538,773,630
Llu00006,136,187,129,125,92,104,85,111,65,36,⋯,109,198,93,113,61,190,183,117,227,229
Llu00007,1021,1044,1379,544,1289,1428,929,968,2103,1562,⋯,1243,1574,2072,1209,1005,2025,3296,1794,2843,2473
Llu00008,1601,1643,2245,1621,2248,2882,1948,2229,2826,2114,⋯,1556,1897,2561,1392,1460,1300,2308,1268,1952,1700
Llu00009,44,18,40,9,30,30,27,9,180,201,⋯,48,143,162,161,159,3,19,0,12,17
Llu00010,576,752,639,483,481,562,555,655,547,260,⋯,463,631,490,433,369,541,303,317,419,369


In [ ]:
samples

In [32]:
#DESEQ2
#COUNDATA CONTIENE CUENTAS CRUDAS, Y EL "FINAL" ES PORQUE SE EXTRAJO UNA LISTA DE GENES NO EXPRESADOS, EN PASOS ANTERIORES
#Esta fórmula especifica el modelo lineal que DESeq2 usará para ajustar los datos. En este caso:
#genotype: Efecto principal del genotipo.
#hpi: Efecto principal del tiempo post-infección.
#hpi:genotype: Interacción entre ambos factores. Esta parte es crucial si quieres saber si la respuesta a la infección cambia dependiendo del genotipo.
dds <- DESeqDataSetFromMatrix(countData=d$counts, 
                              colData=samples, 
                              design= ~ genotype  + hpi + hpi:genotype)

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


In [33]:
dds$genotype<-as.factor(dds$genotype)
dds$hpi<-as.factor(dds$hpi)
dds$tissue<-as.factor(dds$tissue)

In [34]:
#crea una nueva variable group que representa la combinación única de tus condiciones experimentales: hpi + genotype + tissue.
dds$group <- factor(paste0(dds$hpi, dds$genotype, dds$tissue))

In [35]:
#cambia el diseño y compara entre las condiciones de group
#Ya no quiero modelar efectos separados ni interacciones como antes (~ genotype + hpi + hpi:genotype). 
#Ahora quiero que trate cada combinación group como una condición independiente para comparar entre ellas."
design(dds) <- ~ group

In [36]:
#dds será un objeto ahora enriquecido con los resultados del modelo estadístico de DESeq2.
dds <- DESeq(dds,parallel = FALSE) #FALSE PORQUE ES MI NOTEBOOK PERSONAL Y NO PUDE PARALELIZAR

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [37]:
res<-results(dds, cooksCutoff=FALSE, independentFiltering=FALSE)

In [38]:
path<-"/your_path/DESEQ"
dir.create(paste0(path,"/plots"),recursive = FALSE,showWarnings = FALSE)
plots<-paste0(path,"/plots")

dir.create(paste0(path,"/tables"),recursive = FALSE,showWarnings = FALSE)
tables<-paste0(path,"/tables")

In [39]:
levels(dds$group)

[1] "000HPIC195hypocotyl" "000HPIC98hypocotyl"  "024HPIC195hypocotyl"
[4] "024HPIC98hypocotyl"  "060HPIC195hypocotyl" "060HPIC98hypocotyl" 
[7] "084HPIC195hypocotyl" "084HPIC98hypocotyl"

In [40]:
#------------Contrast per genotype------------

# Primero el tratamiento vs el control: C195_000hpi_vs_C98_000hpi
C98_000hpi_vs_C195_000hpi<-results(dds, contrast=c("group","000HPIC98hypocotyl","000HPIC195hypocotyl"))
C98_000hpi_vs_C195_000hpi <- C98_000hpi_vs_C195_000hpi[order(C98_000hpi_vs_C195_000hpi$padj),]
C98_000hpi_vs_C195_000hpi_filtered<-C98_000hpi_vs_C195_000hpi[which(C98_000hpi_vs_C195_000hpi$padj  < 0.01),]
DESEQ_C98_000hpi_vs_C195_000hpi_DE <- subset(C98_000hpi_vs_C195_000hpi_filtered, padj < 0.01 & log2FoldChange>= 2 | padj < 0.01 & log2FoldChange <= -2)


write.table(as.data.frame(C98_000hpi_vs_C195_000hpi), 
            file= paste(tables,"DESEQ_C98_000hpi_vs_C195_000hpi_all.tsv",sep = "/"),
            sep = "\t",quote = F)

write.table(as.data.frame(DESEQ_C98_000hpi_vs_C195_000hpi_DE), 
              file=paste(tables,"DESEQ_C98_000hpi_vs_C195_000hpi_DE.tsv",sep = "/"),
              sep = "\t")

C98_024hpi_vs_C195_024hpi<-results(dds, contrast=c("group","024HPIC98hypocotyl","024HPIC195hypocotyl"))
C98_024hpi_vs_C195_024hpi <- C98_024hpi_vs_C195_024hpi[order(C98_024hpi_vs_C195_024hpi$padj),]
C98_024hpi_vs_C195_024hpi_filtered<-C98_024hpi_vs_C195_024hpi[which(C98_024hpi_vs_C195_024hpi$padj  < 0.01),]
DESEQ_C98_024hpi_vs_C195_024hpi_DE <- subset(C98_024hpi_vs_C195_024hpi_filtered, padj < 0.01 & log2FoldChange>= 2 | padj < 0.01 & log2FoldChange <= -2)

write.table(as.data.frame(C98_024hpi_vs_C195_024hpi), 
            file= paste(tables,"DESEQ_C98_024hpi_vs_C195_024hpi_all.tsv",sep = "/"),
            sep = "\t",quote = F)

write.table(as.data.frame(DESEQ_C98_024hpi_vs_C195_024hpi_DE), 
              file=paste(tables,"DESEQ_C98_024hpi_vs_C195_024hpi_DE.tsv",sep = "/"),
              sep = "\t")


C98_060hpi_vs_C195_060hpi<-results(dds, contrast=c("group","060HPIC98hypocotyl","060HPIC195hypocotyl"))
C98_060hpi_vs_C195_060hpi <- C98_060hpi_vs_C195_060hpi[order(C98_060hpi_vs_C195_060hpi$padj),]
C98_060hpi_vs_C195_060hpi_filtered <-C98_060hpi_vs_C195_060hpi[which(C98_060hpi_vs_C195_060hpi$padj  < 0.01),]
DESEQ_C98_060hpi_vs_C195_060hpi_DE <- subset(C98_060hpi_vs_C195_060hpi_filtered, padj < 0.01 & log2FoldChange>= 2 | padj < 0.01 & log2FoldChange <= -2)

write.table(as.data.frame(C98_060hpi_vs_C195_060hpi), 
            file= paste(tables,"DESEQ_C98_060hpi_vs_C195_060hpi_all.tsv",sep = "/"),
            sep = "\t",quote = F)

write.table(as.data.frame(DESEQ_C98_060hpi_vs_C195_060hpi_DE), 
              file=paste(tables,"DESEQ_C98_060hpi_vs_C195_060hpi_DE.tsv",sep = "/"),
              sep = "\t")

C98_084hpi_vs_C195_084hpi<-results(dds, contrast=c("group","084HPIC98hypocotyl","084HPIC195hypocotyl"))
C98_084hpi_vs_C195_084hpi <- C98_084hpi_vs_C195_084hpi[order(C98_084hpi_vs_C195_084hpi$padj),]
C98_084hpi_vs_C195_084hpi_filtered <-C98_084hpi_vs_C195_084hpi[which(C98_084hpi_vs_C195_084hpi$padj  < 0.01),]
DESEQ_C98_084hpi_vs_C195_084hpi_DE <- subset(C98_084hpi_vs_C195_084hpi_filtered, padj < 0.01 & log2FoldChange>= 2 | padj < 0.01 & log2FoldChange <= -2)

write.table(as.data.frame(C98_084hpi_vs_C195_084hpi), 
            file= paste(tables,"DESEQ_C98_084hpi_vs_C195_084hpi_all.tsv",sep = "/"),
            sep = "\t",quote = F)

write.table(as.data.frame(DESEQ_C98_084hpi_vs_C195_084hpi_DE), 
              file=paste(tables,"DESEQ_C98_084hpi_vs_C195_084hpi_DE.tsv",sep = "/"),
              sep = "\t")

In [41]:
save.image(file = "RNAseq_analysis_EdgeR_DESEQ2_2025_C98vsC195_julio_image.RData")